<a href="https://colab.research.google.com/github/HopeSilkina/deposits_forecast_project/blob/main/notebooks/01_EDA_Modeling_Deposits_Forecast_Ru.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# ПРОГНОЗИРОВАНИЕ ОБЪЕМА ВКЛАДОВ НАСЕЛЕНИЯ В РФ
# Проект для портфолио Data Scientist
# Автор: Надежда Силкина
# Дата: 2026
# ============================================================

# ============================================================
# 1. ПОДКЛЮЧЕНИЕ БИБЛИОТЕК
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tsa.stattools import adfuller
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# Настройка графиков
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Все библиотеки загружены успешно")
print(f"Версия Pandas: {pd.__version__}")

# ============================================================
# 2. ЗАГРУЗКА ДАННЫХ
# ============================================================

url = 'https://raw.githubusercontent.com/HopeSilkina/deposits_forecast_project/main/data/processed_deposits_data.xlsx'
df = pd.read_excel(url, sheet_name='data')

# Преобразование колонки с датами
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

# Сортировка по дате
df.sort_index(inplace=True)

print(f"✅ Данные загружены. Записей: {len(df)}")
print(f"Период: с {df.index.min()} по {df.index.max()}")
print("\nПервые 5 строк:")
print(df.head())

# ============================================================
# 3. РАЗВЕДОЧНЫЙ АНАЛИЗ ДАННЫХ (EDA)
# ============================================================

print("\n" + "="*60)
print("3. РАЗВЕДОЧНЫЙ АНАЛИЗ ДАННЫХ")
print("="*60)

# 3.1. Описательные статистики
print("\n📊 Описательные статистики:")
print(df.describe())

# 3.2. Проверка пропущенных значений
print("\n📊 Пропущенные значения:")
print(df.isnull().sum())

# 3.3. Корреляционная матрица (тепловая карта)
plt.figure(figsize=(12, 10))
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Корреляционная матрица всех признаков', fontsize=14)
plt.tight_layout()
plt.savefig('01_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# 3.4. Визуализация временных рядов
fig, axes = plt.subplots(nrows=5, ncols=2, figsize=(14, 12))
axes = axes.flatten()

# Выбираем только исходные признаки для временных рядов
original_features = ['DEPOS', 'WAGE', 'SERV', 'DEP1', 'CRED1', 'CPI', 'USDind', 'UNEM', 'IPI', 'IMP']

for idx, col in enumerate(original_features):
    ax = axes[idx]
    ax.plot(df.index, df[col], linewidth=1.5)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')
    ax.grid(True, alpha=0.3)

# Удаляем пустые подграфики (если есть)
for idx in range(len(original_features), len(axes)):
    fig.delaxes(axes[idx])

plt.suptitle('Временные ряды всех показателей', fontsize=14, y=0.98)
plt.tight_layout()
plt.savefig('01_time_series_all.png', dpi=300, bbox_inches='tight')
plt.show()

# 3.5. Диаграммы рассеяния: DEPOS vs каждый исходный признак
print("\n📊 Построение диаграмм рассеяния: DEPOS vs каждый предиктор...")

# Исходные признаки (исключая DEPOS)
predictors = ['WAGE', 'SERV', 'DEP1', 'CRED1', 'CPI', 'USDind', 'UNEM', 'IPI', 'IMP']
n_features = len(predictors)

# Расчет размера сетки (3x3 = 9)
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(15, 5 * n_rows))
axes = axes.flatten()

for idx, col in enumerate(predictors):
    ax = axes[idx]
    ax.scatter(df[col], df['DEPOS'], alpha=0.6, s=30, color='steelblue', edgecolor='white')

    # Добавление линии тренда (линейная регрессия)
    mask = ~(df[col].isna() | df['DEPOS'].isna())
    if mask.sum() > 1:
        z = np.polyfit(df.loc[mask, col], df.loc[mask, 'DEPOS'], 1)
        p = np.poly1d(z)
        x_sorted = np.sort(df.loc[mask, col])
        ax.plot(x_sorted, p(x_sorted), "r--", linewidth=1.5,
                label=f'R² = {np.corrcoef(df.loc[mask, col], df.loc[mask, "DEPOS"])[0,1]**2:.3f}')

    ax.set_xlabel(col, fontsize=10)
    ax.set_ylabel('DEPOS', fontsize=10)
    ax.set_title(f'DEPOS vs {col}', fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# Удаляем пустые подграфики (если есть)
for idx in range(n_features, len(axes)):
    fig.delaxes(axes[idx])

plt.suptitle('Диаграммы рассеяния: объем вкладов vs макроэкономические показатели',
             fontsize=14, y=0.98)
plt.tight_layout()
plt.savefig('01_scatter_plots.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Диаграммы рассеяния сохранены в '01_scatter_plots.png'")

# 3.6. Распределение целевой переменной
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['DEPOS'], bins=20, edgecolor='black', alpha=0.7)
axes[0].set_title('Гистограмма: объем вкладов (DEPOS)')
axes[0].set_xlabel('млрд руб.')
axes[0].set_ylabel('Частота')

axes[1].boxplot(df['DEPOS'])
axes[1].set_title('Ящик с усами: объем вкладов (DEPOS)')
axes[1].set_ylabel('млрд руб.')

plt.tight_layout()
plt.savefig('01_depos_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Разведочный анализ завершен")

# ============================================================
# 4. ЭКОНОМЕТРИЧЕСКАЯ ДИАГНОСТИКА
# ============================================================

print("\n" + "="*60)
print("4. ЭКОНОМЕТРИЧЕСКАЯ ДИАГНОСТИКА")
print("="*60)

# 4.1. Тест на стационарность (Augmented Dickey-Fuller)
print("\n🔍 Тест Дики-Фуллера для DEPOS:")

def adf_test(series, series_name):
    result = adfuller(series, autolag='AIC')
    print(f"\n  {series_name}:")
    print(f"    ADF-статистика: {result[0]:.4f}")
    print(f"    p-значение: {result[1]:.4f}")
    print(f"    Критические значения:")
    for key, value in result[4].items():
        print(f"      {key}: {value:.4f}")
    print(f"    Вывод: {'Стационарный ✅' if result[1] < 0.05 else 'Нестационарный ❌'}")

adf_test(df['DEPOS'], 'DEPOS')

# 4.2. Мультиколлинеарность (VIF)
print("\n🔍 Мультиколлинеарность (VIF):")

X_vif = df.drop('DEPOS', axis=1)
# Добавляем константу для VIF
X_vif_with_const = sm.add_constant(X_vif)

vif_data = pd.DataFrame()
vif_data['feature'] = X_vif.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

print(vif_data.sort_values('VIF', ascending=False))
print("\n📌 Интерпретация: VIF > 10 указывает на сильную мультиколлинеарность")

# ============================================================
# 5. ПОДГОТОВКА ДАННЫХ ДЛЯ МОДЕЛИРОВАНИЯ
# ============================================================

print("\n" + "="*60)
print("5. ПОДГОТОВКА ДАННЫХ ДЛЯ МОДЕЛИРОВАНИЯ")
print("="*60)

# 5.1. Логарифмическое преобразование целевой переменной
df['DEPOS_log'] = np.log(df['DEPOS'])

# 5.2. Добавление лаговых признаков
for lag in [1, 3, 6, 12]:
    df[f'DEPOS_lag_{lag}'] = df['DEPOS'].shift(lag)

# 5.3. Подготовка признаков и целевой переменной
X = df.drop(['DEPOS', 'DEPOS_log'], axis=1).dropna()
y = df.loc[X.index, 'DEPOS']

print(f"📊 Размерность X: {X.shape}")
print(f"📊 Размерность y: {y.shape}")

# 5.4. Разделение на обучающую и тестовую выборки (последние 12 месяцев для теста)
train_size = len(X) - 12
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

print(f"\n📊 Обучающая выборка: {len(X_train)} записей")
print(f"📊 Тестовая выборка: {len(X_test)} записей")
print(f"📊 Период теста: {X.index[train_size]} — {X.index[-1]}")

# 5.5. Масштабирование признаков (для Ridge-регрессии)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ============================================================
# 6. ОБУЧЕНИЕ И СРАВНЕНИЕ МОДЕЛЕЙ
# ============================================================

print("\n" + "="*60)
print("6. ОБУЧЕНИЕ И СРАВНЕНИЕ МОДЕЛЕЙ")
print("="*60)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    print(f"\n🔧 Обучение {name}...")

    # Используем масштабированные данные для Ridge, оригинальные для остальных
    if name == 'Ridge Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    # Метрики качества
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

    results[name] = {
        'R²': r2,
        'MAE': mae,
        'RMSE': rmse,
        'MAPE': f"{mape:.2f}%",
        'predictions': y_pred,
        'model': model
    }

    print(f"  ✅ R²: {r2:.4f}")
    print(f"  ✅ MAE: {mae:.2f} млрд руб.")
    print(f"  ✅ RMSE: {rmse:.2f} млрд руб.")
    print(f"  ✅ MAPE: {mape:.2f}%")

# ============================================================
# 6.1. СРАВНИТЕЛЬНАЯ ТАБЛИЦА (ТЕСТОВАЯ ВЫБОРКА)
# ============================================================

results_df = pd.DataFrame({
    name: {k: v for k, v in res.items() if k not in ['predictions', 'model']}
    for name, res in results.items()
}).T

print("\n" + "="*60)
print("📊 СРАВНИТЕЛЬНАЯ ТАБЛИЦА МОДЕЛЕЙ (ТЕСТОВАЯ ВЫБОРКА):")
print("="*60)
print(results_df.round(4))

# ============================================================
# 6.2. ПОЛНЫЕ МЕТРИКИ ЛУЧШЕЙ МОДЕЛИ (ОБУЧАЮЩАЯ ВЫБОРКА)
# ============================================================

print("\n" + "="*60)
print("6.2. ПОЛНЫЕ МЕТРИКИ ЛУЧШЕЙ МОДЕЛИ (ОБУЧАЮЩАЯ + ТЕСТОВАЯ)")
print("="*60)

# Определяем лучшую модель по R² на тесте
best_model_name = results_df['R²'].idxmax()
best_model_results = results[best_model_name]
best_model = best_model_results['model']

# Предсказания на обучающей выборке
if 'Ridge' in best_model_name:
    y_train_pred_best = best_model.predict(X_train_scaled)
else:
    y_train_pred_best = best_model.predict(X_train)

# Предсказания на тестовой выборке
y_test_pred_best = best_model_results['predictions']

# Метрики на обучающей выборке
n_train = len(y_train)
n_params = X_train.shape[1] + 1  # +1 для intercept (intercept есть во всех моделях)

r2_train_best = r2_score(y_train, y_train_pred_best)

# R²_adj на обучающей выборке
if n_train - n_params - 1 > 0:
    r2_adj_train_best = 1 - (1 - r2_train_best) * (n_train - 1) / (n_train - n_params - 1)
else:
    r2_adj_train_best = np.nan

# AIC и BIC на обучающей выборке
residuals_train_best = y_train.values - y_train_pred_best
rss_train_best = np.sum(residuals_train_best**2)

# Unbiased estimator для дисперсии (важно для корректного AIC/BIC)
sigma2_best = rss_train_best / (n_train - n_params)
log_likelihood_best = -0.5 * n_train * (np.log(2 * np.pi * sigma2_best) + 1)

aic_best = -2 * log_likelihood_best + 2 * n_params
bic_best = -2 * log_likelihood_best + n_params * np.log(n_train)

# Метрики на тестовой выборке (уже рассчитаны)
r2_test_best = best_model_results['R²']
rmse_test_best = best_model_results['RMSE']
mae_test_best = best_model_results['MAE']

print(f"\n📊 {best_model_name}:")
print(f"   ОБУЧАЮЩАЯ выборка (n={n_train}):")
print(f"     R²_train = {r2_train_best:.4f}")
print(f"     R²_adj_train = {r2_adj_train_best:.4f}")
print(f"     AIC = {aic_best:.2f}")
print(f"     BIC = {bic_best:.2f}")
print(f"     RSS = {rss_train_best:.2f}")
print(f"   ТЕСТОВАЯ выборка (n={len(y_test)}):")
print(f"     R²_test = {r2_test_best:.4f}")
print(f"     RMSE = {rmse_test_best:.2f} млрд руб.")
print(f"     MAE = {mae_test_best:.2f} млрд руб.")

# Проверка на переобучение
r2_diff = r2_train_best - r2_test_best
if r2_diff > 0.1:
    print(f"\n   ⚠️ Разница R²_train - R²_test = {r2_diff:.4f} — возможно переобучение")
elif r2_diff > 0.05:
    print(f"\n   ℹ️ Разница R²_train - R²_test = {r2_diff:.4f} — умеренная")
else:
    print(f"\n   ✅ Разница R²_train - R²_test = {r2_diff:.4f} — модель хорошо обобщает")

# Сохраняем для использования в следующих блоках
best_model_metrics = {
    'name': best_model_name,
    'r2_train': r2_train_best,
    'r2_adj_train': r2_adj_train_best,
    'aic': aic_best,
    'bic': bic_best,
    'r2_test': r2_test_best,
    'rmse_test': rmse_test_best,
    'mae_test': mae_test_best,
    'n_params': n_params,
    'n_train': n_train,
    'y_train_pred': y_train_pred_best,
    'y_test_pred': y_test_pred_best
}

# ============================================================
# 6.3. ДИАГНОСТИКА ОСТАТКОВ ЛУЧШЕЙ МОДЕЛИ (ОБУЧАЮЩАЯ ВЫБОРКА)
# ============================================================

print("\n" + "="*60)
print("6.3. ДИАГНОСТИКА ОСТАТКОВ (ОБУЧАЮЩАЯ ВЫБОРКА)")
print("="*60)

from scipy.stats import shapiro
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_breusch_godfrey
from statsmodels.graphics.tsaplots import plot_acf

# Остатки на обучающей выборке
residuals_train = y_train.values - y_train_pred_best

# 6.3.1. Тест на нормальность (Шапиро-Уилк)
if 3 <= n_train <= 5000:  # Shapiro-Wilk работает для выборок такого размера
    shapiro_stat, shapiro_p = shapiro(residuals_train)
    normality_ok = shapiro_p > 0.05
else:
    shapiro_stat, shapiro_p = np.nan, np.nan
    normality_ok = None

# 6.3.2. Тест на гомоскедастичность (Бройш-Паган)
try:
    exog_bp = sm.add_constant(y_train_pred_best.reshape(-1, 1))
    bp_stat, bp_p, bp_f, bp_f_p = het_breuschpagan(residuals_train, exog_bp)
    homoscedasticity_ok = bp_p > 0.05
except Exception as e:
    bp_stat, bp_p = np.nan, np.nan
    homoscedasticity_ok = None
    print(f"   ⚠️ Ошибка в тесте Бройша-Пагана: {e}")

# 6.3.3. Тест на автокорреляцию (Бройш-Годфри) — вместо DW
# Преимущества перед DW:
# - Работает при наличии лаговых зависимых переменных
# - Позволяет тестировать автокорреляцию разных порядков
try:
    exog_bg = sm.add_constant(y_train_pred_best.reshape(-1, 1))
    bg_model = sm.OLS(residuals_train, exog_bg).fit()

    # Тест на автокорреляцию 1-го порядка
    bg_stat_1, bg_p_1, _, _ = acorr_breusch_godfrey(bg_model, nlags=1)
    autocorr_1_ok = bg_p_1 > 0.05

    # Тест на автокорреляцию 4-го порядка (сезонная)
    bg_stat_4, bg_p_4, _, _ = acorr_breusch_godfrey(bg_model, nlags=4)
    autocorr_4_ok = bg_p_4 > 0.05
except Exception as e:
    bg_stat_1, bg_p_1 = np.nan, np.nan
    bg_stat_4, bg_p_4 = np.nan, np.nan
    autocorr_1_ok = None
    autocorr_4_ok = None
    print(f"   ⚠️ Ошибка в тесте Бройша-Годфри: {e}")

# Вывод результатов диагностики
print(f"\n🔍 Результаты диагностики для {best_model_name}:")
print(f"   Нормальность остатков:")
print(f"     Shapiro-Wilk stat = {shapiro_stat:.4f}, p = {shapiro_p:.4f}")
print(f"     {'✅ Остатки нормальны' if normality_ok else '⚠️ Отклонение от нормальности' if normality_ok is not None else 'N/A'}")
print(f"   Гомоскедастичность:")
print(f"     Breusch-Pagan stat = {bp_stat:.4f}, p = {bp_p:.4f}")
print(f"     {'✅ Гомоскедастичность' if homoscedasticity_ok else '⚠️ Гетероскедастичность' if homoscedasticity_ok is not None else 'N/A'}")
print(f"   Автокорреляция (тест Бройша-Годфри):")
print(f"     Lag 1: stat = {bg_stat_1:.4f}, p = {bg_p_1:.4f} → {'✅ Нет автокорреляции' if autocorr_1_ok else '⚠️ Автокорреляция' if autocorr_1_ok is not None else 'N/A'}")
print(f"     Lag 4: stat = {bg_stat_4:.4f}, p = {bg_p_4:.4f} → {'✅ Нет автокорреляции' if autocorr_4_ok else '⚠️ Автокорреляция' if autocorr_4_ok is not None else 'N/A'}")

# Сохраняем результаты диагностики
best_model_diagnostics = {
    'shapiro_p': shapiro_p,
    'normality': normality_ok,
    'bp_p': bp_p,
    'homoscedasticity': homoscedasticity_ok,
    'bg_p_1': bg_p_1,
    'autocorr_1': not autocorr_1_ok if autocorr_1_ok is not None else None,
    'bg_p_4': bg_p_4,
    'autocorr_4': not autocorr_4_ok if autocorr_4_ok is not None else None
}

# ============================================================
# 6.4. ГРАФИКИ ДИАГНОСТИКИ ОСТАТКОВ
# ============================================================

print("\n" + "="*60)
print("6.4. ГРАФИКИ ДИАГНОСТИКИ ОСТАТКОВ")
print("="*60)

from scipy.stats import norm

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Диагностика остатков — {best_model_name}\n(обучающая выборка, n={n_train})',
             fontsize=14, fontweight='bold')

# График 1: Гистограмма остатков с кривой нормального распределения
ax = axes[0, 0]
ax.hist(residuals_train, bins=20, edgecolor='black', alpha=0.7, density=True, color='steelblue')
x_range = np.linspace(residuals_train.min(), residuals_train.max(), 100)
ax.plot(x_range, norm.pdf(x_range, residuals_train.mean(), residuals_train.std()),
        'r-', linewidth=2, label='Норм. распределение')
ax.axvline(x=0, color='red', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('Остатки (млрд руб.)')
ax.set_ylabel('Плотность')
ax.set_title('Распределение остатков')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# График 2: Q-Q plot
ax = axes[0, 1]
from scipy import stats as sp_stats
sp_stats.probplot(residuals_train, dist="norm", plot=ax)
ax.set_title('Q-Q plot (проверка нормальности)')
ax.grid(True, alpha=0.3)

# График 3: Остатки vs предсказанные значения
ax = axes[1, 0]
ax.scatter(y_train_pred_best, residuals_train, alpha=0.6, color='steelblue', edgecolors='white')
ax.axhline(y=0, color='red', linestyle='--', linewidth=1.5)
ax.set_xlabel('Предсказанные значения (млрд руб.)')
ax.set_ylabel('Остатки (млрд руб.)')
ax.set_title('Остатки vs Предсказанные\n(проверка гомоскедастичности)')
ax.grid(True, alpha=0.3)

# График 4: Автокорреляционная функция (ACF)
ax = axes[1, 1]
plot_acf(residuals_train, lags=min(20, n_train//4), ax=ax, title='')
ax.set_title('Автокорреляционная функция остатков (ACF)')
ax.set_xlabel('Лаг')
ax.set_ylabel('Автокорреляция')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('01_diagnostics_best_model.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Графики диагностики сохранены в '01_diagnostics_best_model.png'")

# ============================================================
# 6.5. СРАВНЕНИЕ ДИАГНОСТИКИ: DW vs БРОЙШ-ГОДФРИ
# ============================================================

print("\n" + "="*60)
print("6.5. СРАВНЕНИЕ: DW vs ТЕСТ БРОЙША-ГОДФРИ")
print("="*60)

# Для сравнения рассчитываем DW (хотя он некорректен при лаговых переменных)
dw_stat = durbin_watson(residuals_train)

print(f"\n📊 Сравнение тестов на автокорреляцию:")
print(f"   Тест Дарбина-Уотсона (DW):")
print(f"     DW = {dw_stat:.3f}")
print(f"     {'✅ Нет автокорреляции (1.5 < DW < 2.5)' if 1.5 < dw_stat < 2.5 else '⚠️ Автокорреляция'}")
print(f"   Тест Бройша-Годфри (lag 1):")
print(f"     p = {bg_p_1:.4f}")
print(f"     {'✅ Нет автокорреляции' if autocorr_1_ok else '⚠️ Автокорреляция'}")
print(f"\n📌 Почему тест Бройша-Годфри предпочтительнее:")
print(f"   1. DW некорректен при наличии лаговых зависимых переменных")
print(f"   2. BG позволяет тестировать автокорреляцию разных порядков")
print(f"   3. BG более мощный для малых выборок")
print(f"   → В дальнейшем анализе опираемся на тест Бройша-Годфри")

# ============================================================
# 6.6. КОЭФФИЦИЕНТЫ RIDGE-РЕГРЕССИИ (Интерпретируемость)
# ============================================================

print("\n" + "="*60)
print("6.6. КОЭФФИЦИЕНТЫ RIDGE-РЕГРЕССИИ")
print("="*60)

ridge_model = results['Ridge Regression']['model']
feature_names = X.columns

# Получаем коэффициенты (на масштабированных признаках)
coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': ridge_model.coef_
}).sort_values('coefficient', ascending=False)

print("\n📊 Коэффициенты Ridge-регрессии (масштабированные признаки):")
print(coef_df.to_string(index=False))

print("\n📌 Интерпретация:")
print("   Положительный коэффициент → DEPOS УВЕЛИЧИВАЕТСЯ при росте фактора")
print("   Отрицательный коэффициент → DEPOS УМЕНЬШАЕТСЯ при росте фактора")

print("\n🔺 ТОП-5 ДРАЙВЕРОВ (увеличивают вклады):")
print(coef_df.head(5).to_string(index=False))

print("\n🔻 ТОП-5 ДРАГГЕРОВ (уменьшают вклады):")
print(coef_df.tail(5).to_string(index=False))

# ============================================================
# 6.7. ПОДБОР ОПТИМАЛЬНОЙ АЛЬФЫ ДЛЯ RIDGE-РЕГРЕССИИ
# ============================================================

print("\n" + "="*60)
print("6.7. ПОДБОР ОПТИМАЛЬНОЙ АЛЬФЫ ДЛЯ RIDGE-РЕГРЕССИИ")
print("="*60)

from sklearn.linear_model import RidgeCV

# Перебираем диапазон значений alpha
alphas = np.logspace(-3, 3, 50)  # от 0.001 до 1000

# RidgeCV с кросс-валидацией (5 фолдов)
ridge_cv = RidgeCV(alphas=alphas, scoring='neg_mean_squared_error', cv=5)
ridge_cv.fit(X_train_scaled, y_train)

best_alpha = ridge_cv.alpha_
print(f"\n✅ Лучшая alpha по кросс-валидации: {best_alpha:.4f}")

# Обучение модели с лучшей alpha
ridge_optimal = Ridge(alpha=best_alpha)
ridge_optimal.fit(X_train_scaled, y_train)
y_pred_optimal = ridge_optimal.predict(X_test_scaled)

# Сравнение с исходной Ridge (alpha=1.0)
r2_original = results['Ridge Regression']['R²']
r2_optimal = r2_score(y_test, y_pred_optimal)
rmse_original = results['Ridge Regression']['RMSE']
rmse_optimal = np.sqrt(mean_squared_error(y_test, y_pred_optimal))

print(f"\n📊 Сравнение:")
print(f"   Исходная Ridge (alpha=1.0):  R² = {r2_original:.4f}, RMSE = {rmse_original:.2f}")
print(f"   Оптимальная Ridge (alpha={best_alpha:.4f}): R² = {r2_optimal:.4f}, RMSE = {rmse_optimal:.2f}")

# Если оптимальная лучше, обновляем лучшую модель
if r2_optimal > r2_original:
    print("\n✅ Оптимальная Ridge превосходит исходную! Обновляем лучшую модель...")
    results['Ridge Regression (Optimal)'] = {
        'R²': r2_optimal,
        'MAE': mean_absolute_error(y_test, y_pred_optimal),
        'RMSE': rmse_optimal,
        'MAPE': f"{np.mean(np.abs((y_test - y_pred_optimal) / y_test)) * 100:.2f}%",
        'predictions': y_pred_optimal,
        'model': ridge_optimal
    }
    best_model_name = 'Ridge Regression (Optimal)'
    best_pred = y_pred_optimal
else:
    print("\nℹ️ Исходная Ridge (alpha=1.0) уже оптимальна.")

# ============================================================
# 6.8. ОТБОР ПРИЗНАКОВ ДЛЯ ЛИНЕЙНОЙ РЕГРЕССИИ (p-значения)
# ============================================================

print("\n" + "="*60)
print("6.8. ОТБОР ПРИЗНАКОВ ДЛЯ ЛИНЕЙНОЙ РЕГРЕССИИ")
print("="*60)

# Добавляем константу для statsmodels
X_train_const = sm.add_constant(X_train)
X_test_const = sm.add_constant(X_test)

# Оценка OLS с помощью statsmodels
ols_full = sm.OLS(y_train, X_train_const).fit()

print("\n📊 Полная сводка линейной регрессии (p-значения):")
print("="*60)
print(ols_full.summary())

# Извлечение p-значений
p_values = ols_full.pvalues
significant_features = p_values[p_values < 0.05].index.tolist()

# Удаляем 'const' из списка
significant_features = [f for f in significant_features if f != 'const']

print(f"\n📊 Признаки с p-значением < 0.05 (статистически значимые):")
print(f"   {len(significant_features)} признаков: {significant_features}")

# Построение сокращенной модели только со значимыми признаками
if len(significant_features) > 0:
    X_train_reduced = X_train[significant_features]
    X_test_reduced = X_test[significant_features]

    # Обучение сокращенной линейной регрессии
    lr_reduced = LinearRegression()
    lr_reduced.fit(X_train_reduced, y_train)
    y_pred_reduced = lr_reduced.predict(X_test_reduced)

    # Метрики
    r2_reduced = r2_score(y_test, y_pred_reduced)
    rmse_reduced = np.sqrt(mean_squared_error(y_test, y_pred_reduced))
    mae_reduced = mean_absolute_error(y_test, y_pred_reduced)

    print(f"\n📊 Сокращенная линейная регрессия (только значимые признаки):")
    print(f"   R² = {r2_reduced:.4f}")
    print(f"   RMSE = {rmse_reduced:.2f} млрд руб.")
    print(f"   MAE = {mae_reduced:.2f} млрд руб.")

    # Сравнение с исходной линейной регрессией
    r2_original_lr = results['Linear Regression']['R²']
    rmse_original_lr = results['Linear Regression']['RMSE']

    print(f"\n📊 Сравнение с полной линейной регрессией:")
    print(f"   Полная модель:   R² = {r2_original_lr:.4f}, RMSE = {rmse_original_lr:.2f}")
    print(f"   Сокращенная модель: R² = {r2_reduced:.4f}, RMSE = {rmse_reduced:.2f}")

    if r2_reduced > r2_original_lr:
        print("   ✅ Сокращенная модель лучше (удален шум)")
    else:
        print("   ℹ️ Полная модель лучше (все признаки вносят вклад)")

    # Вывод коэффициентов сокращенной модели
    coef_reduced = pd.DataFrame({
        'feature': significant_features,
        'coefficient': lr_reduced.coef_
    }).sort_values('coefficient', ascending=False)

    print("\n📊 Коэффициенты сокращенной модели:")
    print(coef_reduced.to_string(index=False))

else:
    print("\n⚠️ Нет признаков с p-значением < 0.05. Все признаки незначимы.")

print("\n✅ Диагностика моделей завершена")

# ============================================================
# 7. ВИЗУАЛИЗАЦИЯ ПРОГНОЗА
# ============================================================

print("\n" + "="*60)
print("7. ВИЗУАЛИЗАЦИЯ ПРОГНОЗА")
print("="*60)

# 7.1. График исторических данных vs прогноз
plt.figure(figsize=(14, 7))

# Исторические данные (все)
plt.plot(df.index, df['DEPOS'], label='Фактические данные', color='#1f77b4', linewidth=2.5)

# Прогноз лучшей модели на тестовом периоде
best_pred = results[best_model_name]['predictions']
test_dates = X.index[train_size:]

plt.plot(test_dates, best_pred, label=f'Прогноз ({best_model_name})',
         color='#ff7f0e', linestyle='--', linewidth=2.5)

# Доверительный интервал (±2 RMSE)
rmse_best = results[best_model_name]['RMSE']
plt.fill_between(test_dates,
                 best_pred - 2*rmse_best,
                 best_pred + 2*rmse_best,
                 alpha=0.25, color='#ff7f0e', label='95% доверительный интервал')

plt.title('Прогнозирование объема вкладов населения в РФ\n' +
          f'Лучшая модель: {best_model_name} (R² = {results_df.loc[best_model_name, "R²"]:.4f})',
          fontsize=14)
plt.xlabel('Дата')
plt.ylabel('Объем вкладов, млрд руб.')
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('01_forecast_plot.png', dpi=300, bbox_inches='tight')
plt.show()

# 7.2. Фактические vs Прогнозные (диаграмма рассеяния)
plt.figure(figsize=(8, 8))

all_y = pd.concat([y_train, y_test])
all_pred = pd.concat([
    pd.Series(results[best_model_name]['model'].predict(X_train_scaled if best_model_name == 'Ridge Regression' else X_train),
              index=y_train.index),
    pd.Series(best_pred, index=y_test.index)
])

plt.scatter(all_y, all_pred, alpha=0.6)
plt.plot([all_y.min(), all_y.max()], [all_y.min(), all_y.max()],
         'r--', linewidth=2, label='Идеальная линия')
plt.xlabel('Фактические значения')
plt.ylabel('Прогнозные значения')
plt.title(f'Фактические vs Прогнозные ({best_model_name})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('01_actual_vs_predicted.png', dpi=300, bbox_inches='tight')
plt.show()

# 7.3. Важность признаков (только для Random Forest)
if 'Random Forest' in results:
    rf_model = results['Random Forest']['model']
    importance = pd.DataFrame({
        'feature': X.columns,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)

    plt.figure(figsize=(10, 8))
    plt.barh(importance['feature'], importance['importance'], color='steelblue')
    plt.xlabel('Важность')
    plt.title('Важность признаков (Random Forest)')
    plt.tight_layout()
    plt.savefig('01_feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()

    print("\n📊 ТОП-5 НАИБОЛЕЕ ВАЖНЫХ ПРИЗНАКОВ:")
    print(importance.head(5))

# ============================================================
# 8. ЭКСПОРТ РЕЗУЛЬТАТОВ
# ============================================================

print("\n" + "="*60)
print("8. ЭКСПОРТ РЕЗУЛЬТАТОВ")
print("="*60)

# Создание DataFrame с результатами прогноза
forecast_df = pd.DataFrame({
    'Date': test_dates,
    'Actual': y_test,
    f'Predicted_{best_model_name}': best_pred,
    'Lower_95%_CI': best_pred - 2*rmse_best,
    'Upper_95%_CI': best_pred + 2*rmse_best
})

# Сохранение в CSV
forecast_df.to_csv('01_forecast_results.csv', index=False)
print("✅ Результаты прогноза сохранены в '01_forecast_results.csv'")

# Сохранение метрик
results_df.to_csv('01_model_metrics.csv')
print("✅ Метрики моделей сохранены в '01_model_metrics.csv'")

# ============================================================
# 9. ИТОГОВЫЙ ВЫВОД
# ============================================================

print("\n" + "="*60)
print("9. ИТОГОВЫЙ ВЫВОД")
print("="*60)

print(f"""
📌 КЛЮЧЕВЫЕ РЕЗУЛЬТАТЫ БЛОКА 1:

1. ЛУЧШАЯ МОДЕЛЬ: {best_model_name}
   ┌─────────────────────────────────────────────────────────┐
   │ ОБУЧАЮЩАЯ выборка (n={n_train}):                       │
   │   R²_train = {r2_train_best:.4f}                       │
   │   R²_adj_train = {r2_adj_train_best:.4f}               │
   │   AIC = {aic_best:.2f}                                 │
   │   BIC = {bic_best:.2f}                                 │
   ├─────────────────────────────────────────────────────────┤
   │ ТЕСТОВАЯ выборка (n={len(y_test)}):                    │
   │   R²_test = {r2_test_best:.4f}                         │
   │   RMSE = {rmse_test_best:.2f} млрд руб.               │
   │   MAE = {mae_test_best:.2f} млрд руб.                 │
   └─────────────────────────────────────────────────────────┘

2. ДИАГНОСТИКА ОСТАТКОВ (обучающая выборка):
   - Нормальность: {'✅' if normality_ok else '⚠️ Отклонение' if normality_ok is not None else 'N/A'}
   - Гомоскедастичность: {'✅' if homoscedasticity_ok else '⚠️ Гетероскедастичность' if homoscedasticity_ok is not None else 'N/A'}
   - Автокорреляция (lag 1): {'✅ Нет' if autocorr_1_ok else '⚠️ Есть' if autocorr_1_ok is not None else 'N/A'}
   - Автокорреляция (lag 4): {'✅ Нет' if autocorr_4_ok else '⚠️ Есть' if autocorr_4_ok is not None else 'N/A'}

3. КОЭФФИЦИЕНТЫ RIDGE-РЕГРЕССИИ (ТОП-ДРАЙВЕРЫ):
""")

# Получение коэффициентов из лучшей модели (если Ridge)
if 'Ridge' in best_model_name:
    best_coef_df = pd.DataFrame({
        'feature': feature_names,
        'coefficient': best_model.coef_
    }).sort_values('coefficient', ascending=False)

    print("   🔺 ТОП-3 ДРАЙВЕРА (увеличивают вклады):")
    for _, row in best_coef_df.head(3).iterrows():
        print(f"      - {row['feature']}: {row['coefficient']:.4f}")

    print("   🔻 ТОП-3 ДРАГГЕРА (уменьшают вклады):")
    negative_coefs = best_coef_df[best_coef_df['coefficient'] < 0].sort_values('coefficient')
    for _, row in negative_coefs.head(3).iterrows():
        print(f"      - {row['feature']}: {row['coefficient']:.4f}")

print(f"""
4. КЛЮЧЕВЫЕ ВЫВОДЫ:
   - Ridge-регрессия с alpha=1.0 показала наилучшие результаты на тесте
   - Мультиколлинеарность (VIF > 100) успешно обработана регуляризацией
   - Лаговые признаки (1, 3, 6, 12 месяцев) критически важны для прогноза
   - Разница R²_train - R²_test = {r2_diff:.4f} {'(возможно переобучение)' if r2_diff > 0.1 else '(хорошее обобщение)'}
   - Автокорреляция {'4-го порядка требует внимания' if autocorr_4_ok is not None and not autocorr_4_ok else '1-го порядка требует внимания' if autocorr_1_ok is not None and not autocorr_1_ok else 'не обнаружена'}

5. СЛЕДУЮЩИЕ ШАГИ:
   - [x] Базовая модель построена и продиагностирована
   - [ ] Feature Engineering: добавить сезонные и макроэкономические признаки (Блок 4)
   - [ ] Сравнить с полной моделью по всем метрикам (R²_train, AIC, BIC)
   - [ ] Учесть автокорреляцию остатков (при необходимости)
   - [ ] Протестировать SARIMA/Prophet для сравнения

6. СОХРАНЕННЫЕ ФАЙЛЫ:
   - correlation_matrix.png — корреляционная матрица
   - time_series_all.png — временные ряды всех показателей
   - scatter_plots.png — диаграммы рассеяния
   - depos_distribution.png — распределение целевой переменной
   - forecast_plot.png — график прогноза
   - actual_vs_predicted.png — фактические vs прогнозные
   - feature_importance.png — важность признаков (Random Forest)
   - 01_diagnostics_best_model.png — диагностика остатков
   - forecast_results.csv — результаты прогноза
   - model_metrics.csv — метрики моделей
""")

print("✅ БЛОК 1 УСПЕШНО ЗАВЕРШЕН")